Heart Disease Predictor




# Heart Disease Predictor

A binary classification project to predict presence of heart disease
using the Kaggle Heart Disease dataset (Cleveland data).

## Key Finding During EDA
Dataset contained 723 duplicate rows (70% of 1025 rows) — removed
before any processing. All results are on 302 clean, unique records.

## What I Did
- Identified categorical vs numerical features from domain knowledge
- One-hot encoded nominal categoricals (thal, restecg)
- Train-test split → StandardScaler → model training
- Evaluated using recall as primary metric: missing a sick patient
  is worse than a false alarm in medical context
- Hyperparameter tuned KNN via GridSearchCV (scoring=recall)

## Results
| Model | Accuracy | Recall (disease) | Missed Patients |
|---|---|---|---|
| Logistic Regression | 0.74 | 0.83 | 5 |
| KNN (tuned) | 0.74 | 0.79 | 6 |

## Conclusion
Logistic Regression selected — equal accuracy, higher recall,
and greater interpretability for clinical use.

In [170]:
# Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [171]:
import pandas as pd

In [172]:
df = pd.read_csv('/content/drive/MyDrive/april_grind/heart_disease_predictor/heart.csv')

In [173]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [174]:
df = df.drop_duplicates()

In [175]:
print(df.shape)
print(df.columns.to_list())

(302, 14)
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']


In [176]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 302 entries, 0 to 878
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       302 non-null    int64  
 1   sex       302 non-null    int64  
 2   cp        302 non-null    int64  
 3   trestbps  302 non-null    int64  
 4   chol      302 non-null    int64  
 5   fbs       302 non-null    int64  
 6   restecg   302 non-null    int64  
 7   thalach   302 non-null    int64  
 8   exang     302 non-null    int64  
 9   oldpeak   302 non-null    float64
 10  slope     302 non-null    int64  
 11  ca        302 non-null    int64  
 12  thal      302 non-null    int64  
 13  target    302 non-null    int64  
dtypes: float64(1), int64(13)
memory usage: 35.4 KB


In [177]:
df.isnull().sum()

,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [178]:
df.describe()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
count,302.00000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000,302.000000
mean,54.42053,0.682119,0.963576,131.602649,246.500000,0.149007,0.526490,149.569536,0.327815,1.043046,1.397351,0.718543,2.314570,0.543046
std,9.04797,0.466426,1.032044,17.563394,51.753489,0.356686,0.526027,22.903527,0.470196,1.161452,0.616274,1.006748,0.613026,0.498970
min,29.00000,0.000000,0.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,48.00000,0.000000,0.000000,120.000000,211.000000,0.000000,0.000000,133.250000,0.000000,0.000000,1.000000,0.000000,2.000000,0.000000
50%,55.50000,1.000000,1.000000,130.000000,240.500000,0.000000,1.000000,152.500000,0.000000,0.800000,1.000000,0.000000,2.000000,1.000000
75%,61.00000,1.000000,2.000000,140.000000,274.750000,0.000000,1.000000,166.000000,1.000000,1.600000,2.000000,1.000000,3.000000,1.000000
max,77.00000,1.000000,3.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,2.000000,4.000000,3.000000,1.000000


In [179]:
df['target'].value_counts()

,count
target,
1,164
0,138


In [180]:
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


Splitting the data into testing and training then into X and y

In [181]:
X = df.drop(columns='target')
y = df['target']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state=42)

now one hot encoding the thal data

In [182]:
# One-Hot encoding thal column
encoded = pd.get_dummies(df['thal'], drop_first=True, dtype=int)
encoded.columns = ['thal_fixed', 'thal_normal', 'thal_reversable']
df.drop(columns='thal', inplace=True)
df = pd.concat([df, encoded], axis=1)

# One hot encoding restecg
encoded = pd.get_dummies(df['restecg'], drop_first=True, dtype=int)
encoded.columns = ['restecg_st_t_wave', 'restecg_left_ventricular_hypertrophy']
df.drop(columns='restecg', inplace=True)
df = pd.concat([df, encoded], axis=1)

df.head()

,age,sex,cp,trestbps,chol,fbs,thalach,exang,oldpeak,slope,ca,target,thal_fixed,thal_normal,thal_reversable,restecg_st_t_wave,restecg_left_ventricular_hypertrophy
0,52,1,0,125,212,0,168,0,1.0,2,2,0,0,0,1,1,0
1,53,1,0,140,203,1,155,1,3.1,0,0,0,0,0,1,0,0
2,70,1,0,145,174,0,125,1,2.6,0,0,0,0,0,1,1,0
3,61,1,0,148,203,0,161,0,0.0,2,1,0,0,0,1,1,0
4,62,0,0,138,294,1,106,0,1.9,1,3,0,0,1,0,1,0


In [183]:
# Splitting of training and testing data
from sklearn.model_selection import train_test_split

X = df.drop(columns='target')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state=42)

Now comes SCALING the data. scale the training data using fit_transform and the testing data using transform

In [184]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

type(X_train)

numpy.ndarray

Now must we make the model. We will make KNN and Logistic Regression model

In [185]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

logistic_model = LogisticRegression(max_iter = 1000)
logistic_model.fit(X_train,y_train)

y_pred = logistic_model.predict(X_test)

logistic_accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {logistic_accuracy:.2f}\n')

logistic_cm = confusion_matrix(y_test, y_pred)
logistic_report = classification_report(y_test, y_pred)

print(f'Confusion Matrix:\n{logistic_cm}\n')
print(f'Classification Report:\n{logistic_report}')

Accuracy: 0.74

Confusion Matrix:
[[21 11]
 [ 5 24]]

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.66      0.72        32
           1       0.69      0.83      0.75        29

    accuracy                           0.74        61
   macro avg       0.75      0.74      0.74        61
weighted avg       0.75      0.74      0.74        61



In [186]:
from sklearn.neighbors import KNeighborsClassifier

KNN_model = KNeighborsClassifier(n_neighbors = 5)
KNN_model.fit(X_train, y_train)
y_pred = KNN_model.predict(X_test)

knn_accuracy = accuracy_score(y_test, y_pred)
cm_knn = confusion_matrix(y_test, y_pred)
report_knn = classification_report(y_test, y_pred)
print(f"Accuracy: {knn_accuracy:.4f}\n")
print(f"Confusion Matrix:\n{cm_knn}\n")
print(f"Classification Report:\n{report_knn}")

Accuracy: 0.7049

Confusion Matrix:
[[22 10]
 [ 8 21]]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.69      0.71        32
           1       0.68      0.72      0.70        29

    accuracy                           0.70        61
   macro avg       0.71      0.71      0.70        61
weighted avg       0.71      0.70      0.71        61



In [187]:
# Now I shall do Cross_validation for the KNN model

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
parameter_grid = {
    'n_neighbors':[1,3,5,7,9],
    'weights':['uniform', 'distance'],
    'metric':['euclidean', 'manhattan']
}

#Grid Search
grid = GridSearchCV(KNN_model, parameter_grid, cv=5, scoring='recall')
# Train
grid.fit(X_train,y_train)
# Best Settings
print(grid.best_params_)
# Best Score
print(grid.best_score_)

scores = cross_val_score(KNN_model, X, y, cv=5)

print(scores)
print(scores.mean())


{'metric': 'euclidean', 'n_neighbors': 5, 'weights': 'uniform'}
0.8962962962962961
[0.6557377  0.55737705 0.73333333 0.71666667 0.61666667]
0.6559562841530056


In [188]:
#making the KNN model with best parameters found in GridSearchCV
KNN_model_best = KNeighborsClassifier(n_neighbors = 5, metric='euclidean', weights='distance')
KNN_model_best.fit(X_train, y_train)
y_pred = KNN_model_best.predict(X_test)

accuracy_bestknn = accuracy_score(y_test, y_pred)
cm_bestknn = confusion_matrix(y_test, y_pred)
report_bestknn = classification_report(y_test, y_pred)

print(f'Accuracy: {accuracy_bestknn:.4f}\n')
print(f'Confusion Matrix:\n{cm_bestknn}\n')
print(f'Classification Report:\n{report_bestknn}')


Accuracy: 0.7377

Confusion Matrix:
[[22 10]
 [ 6 23]]

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.69      0.73        32
           1       0.70      0.79      0.74        29

    accuracy                           0.74        61
   macro avg       0.74      0.74      0.74        61
weighted avg       0.74      0.74      0.74        61

